In [ ]:
import kagglehub

path = kagglehub.competition_download('dlmmdd-workshop-synthetic-source-attribution-challenge')

print("Path to competition files:", path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image
import cv2
%matplotlib inline

# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# TorchVision
import torchvision.transforms.v2 as v2
from torchvision.models import efficientnet_b4,EfficientNet_B4_Weights
from torchvision.utils import make_grid

# Data utilities
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Evaluation
from sklearn.metrics import confusion_matrix, classification_report,roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
MAIN_PATH = "/kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data"
TRAIN_PATH = os.path.join(MAIN_PATH, 'Training')
TEST_PATH = os.path.join(MAIN_PATH, 'Test')
TRAIN_CSV_PATH = os.path.join(MAIN_PATH, 'training.csv')
TEST_CSV_PATH = os.path.join(MAIN_PATH, 'test.csv')
LABEL_TO_NAMES_PATH = os.path.join(MAIN_PATH, 'sources.txt')

NUM_OF_CLASSES = 10

SIZE = 384
BATCH_SIZE = 64
EPOCHS = 10
PATIENCE = 5
LR = 1e-4

MEAN_NORM = [0.485, 0.456, 0.406]
STD_NORM  = [0.229, 0.224, 0.225]

NUM_WORKERS = 4
PIN_MEMORY = True

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
train_df

In [ ]:
train_df['y'].value_counts()

In [ ]:
def show_all_extensions(path):
    imgs_names = os.listdir(path)
    result=[]
    for img_name in imgs_names:
        x = img_name.split('.')[-1]
        if x not in result:
            result.append(x)
    return result

show_all_extensions(TRAIN_PATH)

In [ ]:
def show_all_imgs_sizes(path):
    imgs_names = os.listdir(path)
    all_sizes=[]
    for img_name in imgs_names:
        img_shape = plt.imread(os.path.join(path,img_name)).shape
        if img_shape not in all_sizes:
            all_sizes.append(img_shape)

    return all_sizes

show_all_imgs_sizes(TRAIN_PATH)

In [ ]:
def show_img_per_class():
    """Display one random image from each class."""
    unique_classes = sorted(train_df['y'].unique())
    num_classes = len(unique_classes)

    cols = min(4, num_classes)
    rows = (num_classes + cols - 1) // cols
    
    plt.figure(figsize=(cols * 4, rows * 4))
    
    for idx, cls in enumerate(unique_classes):
        
        class_df = train_df[train_df['y'] == cls]
        
        img_path = np.random.choice(class_df["path"].values)
        
        img_full_path = os.path.join(TRAIN_PATH, img_path.split('/')[-1])
        img = plt.imread(img_full_path)
        
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(img)
        plt.title(f"Class/Label: {cls}")
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()


show_img_per_class()

In [ ]:
def get_transforms(size=SIZE, apply_on_train=False):
    """
    Build a transform pipeline for training or validation/test.

    Training pipeline includes augmentation (flip, rotation, color jitter)
    to reduce overfitting. Val/test pipeline only resizes and normalizes.

    Parameters
    ----------
    size           : int  — target image size (default 224)
    apply_on_train : bool — True to include augmentation steps

    Returns
    -------
    torchvision.transforms.v2.Compose
    """
    base = [
        v2.Resize((size, size)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ]

    augmentation = [
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomGrayscale(p=0.1),
        v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    ]

    # ImageNet mean/std normalization (required for pretrained models)
    tail = [v2.Normalize(MEAN_NORM, STD_NORM)]

    if apply_on_train:
        return v2.Compose(base + augmentation + tail)
    return v2.Compose(base + tail)

In [ ]:
class SynthiticDataset(Dataset):
    def __init__(self,dataframe,transform = None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        image_path = self.df.iloc[idx,1]
        label = self.df.iloc[idx,2]

        img = Image.open(image_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img,label
        

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
train_df, val_df = train_test_split(
    train_df,
    train_size=0.2,
    random_state=42,
    stratify=TRAIN_CSV_PATH["y"] 
)

In [ ]:
train_dataset = SynthiticDataset(train_df,get_transforms(SIZE,True))
val_dataset = SynthiticDataset(val_df,get_transforms(SIZE))
test_dataset = SynthiticDataset(test_df,get_transforms(SIZE))

In [ ]:
train_dl = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)
val_dl = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)
test_dl = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY)

In [ ]:
def denorm(imgs):
    """
    Reverse ImageNet normalization so images can be displayed correctly.

    Parameters
    ----------
    imgs : torch.Tensor — normalized image batch (B, C, H, W) or single image (C, H, W)

    Returns
    -------
    torch.Tensor in [0, 1] range
    """
    mean = torch.tensor(MEAN_NORM).view(1, 3, 1, 1).to(imgs.device)
    std  = torch.tensor(STD_NORM).view(1, 3, 1, 1).to(imgs.device)

    if imgs.dim() == 3:
        imgs = imgs.unsqueeze(0)

    return imgs * std + mean

In [ ]:
def show_grid_images(dataloader):
    """Display a grid of images from one batch (useful for sanity-checking augmentation)."""
    imgs, labels = next(iter(dataloader))
    plt.figure(figsize=(30, 30))
    plt.imshow(make_grid(denorm(imgs), nrow=16).permute(1, 2, 0))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

show_grid_images(train_dl)

In [ ]:
def replace_classifier(model, name):
    if "EfficientNet" in name:
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.classifier[1].in_features, NUM_OF_CLASSES)
        )
    elif "ConvNeXt" in name:
        model.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(p=0.3),
            nn.Linear(model.classifier[2].in_features, NUM_OF_CLASSES)
        )
    elif "VisionTransformer" in name:
        model.heads = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.heads.in_features, NUM_OF_CLASSES)
        )

    return model

In [ ]:
class SRMConv(nn.Module):
    def __init__(self):
        super().__init__()
        # 3 SRM kernels 
        f1 = np.array([[ 0,  0,  0,  0,  0],
                       [ 0, -1,  2, -1,  0],
                       [ 0,  2, -4,  2,  0],
                       [ 0, -1,  2, -1,  0],
                       [ 0,  0,  0,  0,  0]]) / 4.0

        f2 = np.array([[-1,  2, -2,  2, -1],
                       [ 2, -6,  8, -6,  2],
                       [-2,  8,-12,  8, -2],
                       [ 2, -6,  8, -6,  2],
                       [-1,  2, -2,  2, -1]]) / 12.0

        f3 = np.array([[ 0,  0,  0,  0,  0],
                       [ 0,  0,  0,  0,  0],
                       [ 0,  1, -2,  1,  0],
                       [ 0,  0,  0,  0,  0],
                       [ 0,  0,  0,  0,  0]]) / 2.0

        filters = np.stack([f1, f2, f3], axis=0)          # (3, 5, 5)
        filters = np.expand_dims(filters, axis=1)          # (3, 1, 5, 5)
        filters = np.repeat(filters, 3, axis=1)            # (3, 3, 5, 5)

        self.weight = nn.Parameter(
            torch.tensor(filters, dtype=torch.float32),
            requires_grad=False
        )

    def forward(self, x):
        return nn.functional.conv2d(x, self.weight, padding=2)

In [ ]:
class DualStreamModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.rgb_backbone = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
        rgb_features = self.rgb_backbone.classifier[1].in_features
        self.rgb_backbone.classifier = nn.Identity()
        self.backbone_name = self.rgb_backbone.__class__.__name__
        # Stream 2 — SRM noise
        self.srm = SRMConv()
        self.srm_backbone = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
        self.srm_backbone.classifier = nn.Identity()

        # Fusion head
        self.classifier = nn.Sequential(
            nn.Linear(rgb_features * 2, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        rgb_feat = self.rgb_backbone(x)
        srm_feat = self.srm_backbone(self.srm(x))
        combined = torch.cat([rgb_feat, srm_feat], dim=1)
        return self.classifier(combined)

In [ ]:
model = DualStreamModel(num_classes=NUM_OF_CLASSES).to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    model.backbone_name = model.module.backbone_name
criterion = nn.CrossEntropyLoss()


# Cosine LR Scheduler
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=len(train_dl),
    epochs=EPOCHS
)

# Mixed Precision
scaler = torch.amp.GradScaler(device)

In [ ]:
def fit(model):
    history   = []
    best_acc = 0.0
    counter   = 0

    for epoch in range(EPOCHS):
        train_all_loss = []
        train_all_acc  = []

        model.train()

        for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs, labels = batch
            imgs   = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(device_type='cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            _, preds = torch.max(outputs, dim=1)
            train_all_loss.append(loss.detach().cpu().item())
            train_all_acc.append((preds == labels).float().mean().item())

        train_loss = sum(train_all_loss) / len(train_all_loss)
        train_acc  = sum(train_all_acc)  / len(train_all_acc)

        val_all_loss = []
        val_all_acc  = []

        model.eval()
        with torch.no_grad():
            for batch in val_dl:
                imgs, labels = batch
                imgs   = imgs.to(device)
                labels = labels.to(device)

                outputs = model(imgs)
                loss    = criterion(outputs, labels)

                _, preds = torch.max(outputs, dim=1)
                val_all_loss.append(loss.detach().cpu().item())
                val_all_acc.append((preds == labels).float().mean().item())



        val_loss = sum(val_all_loss) / len(val_all_loss)
        val_acc  = sum(val_all_acc)  / len(val_all_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        history.append({
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss":   val_loss,   "val_acc":   val_acc
        })

        if val_acc > best_acc:
            best_acc = val_acc
            counter   = 0
            torch.save(model.state_dict(), f"best_{model.__class__.__name__}_model.pth")
            print("  >> best model saved")
        else:
            counter += 1
            print(f"  No improvement ({counter}/{PATIENCE})")
            if counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    return history

history = fit(model)